# Fitting a surface, and finding out whether you should believe it

The world below is simulated, which means there is a right answer to compare against — a
luxury no real fit has. That is the point of running it here: the same call, on real data,
returns the same object with the same diagnostics, and this notebook is where you can check
what those diagnostics are worth.

`fit()` hands the `ModelSpec` to a backend. The default is the sampler-free Laplace
approximation — honest by construction: if the mode search does not converge or the Hessian at
the mode is not positive definite, the result is a typed `Unverified`, not a posterior. With the
`[numpyro]` extra, `backend="numpyro"` runs NUTS on the same tree through the jax interpreter.

In [ ]:
import numpy as np

from axiom.core import Interval, is_failure
from axiom.surface import (
    FitResult, GeometricCarryover, HillKernel, MarginalHorizon, Surface, counterfactual_doses, fit, marginal,
    marginal_expr, marginal_total, predict, predict_marginal,
)
from axiom.sim import surface_world

from axiom.display import enable, table

import sys; sys.path[:0] = ["..", "../.."]  # nbs/ is on the path either way
from _style import BLUE, ORANGE, caption, curve_band, intervals, lines

enable();  # every axiom result renders itself from here on

In [ ]:
world = surface_world(n_units=4, n_periods=30, treatments=("a",), kernels=HillKernel(reference_dose=50.0),
                      carryover=GeometricCarryover(max_lag=4), intercept="shared", noise_sd=0.5, seed=7)
print({k: np.round(v, 3) for k, v in world.theta.items()})
print(world.panel)

In [ ]:
from axiom.display import show, table

result: FitResult = fit(world.spec, world.panel, backend="laplace", draws=2000, seed=0)
print("converged:", result.converged, "| provenance keys:", sorted(result.provenance)[:6])
post = result.posterior
if is_failure(post):
    show(post)
else:
    rows = []
    for name in ("beta_a", "k_a", "s_a", "lam_a", "sigma"):
        s = post.summary(name, definition="hdi", mass=0.9)
        iv: Interval = s.interval
        rows.append([name, f"{float(world.theta[name]):.3f}", f"{s.mean:.3f}", str(iv)])
    table(rows, headers=("parameter", "truth", "posterior mean", "90% HDI"))

In [ ]:
if not is_failure(post):
    recovered = []
    for name in ("beta_a", "k_a", "s_a", "lam_a", "sigma"):
        truth = float(world.theta[name])
        s_ = post.summary(name, definition="hdi", mass=0.9)
        recovered.append((f"{name}  (truth {truth:.3g})", s_.mean / truth,
                          s_.interval.lower / truth, s_.interval.upper / truth))
    fig = intervals(
        recovered, ref=1.0, ref_label="truth",
        title="Did it get the world back?",
        subtitle="posterior mean and 90% HDI for every structural parameter, as a fraction of the value that generated the data",
        x_title="posterior ÷ truth",
    )
    caption(fig, "Every interval contains the truth, and the wide ones are wide for a reason: "
                 "amplitude and half-saturation trade off along a ridge — raise both and the "
                 "curve barely moves — while the shape and the carryover are pinned by the "
                 "curvature and the timing, which nothing else can imitate.")
    fig

## Prediction and marginal effects

`predict` evaluates the tree once per draw; `marginal` is the closed-form derivative of the
kernel through the carryover (same-period effect, `w_0 · f'`), `marginal_total` accumulates over
the carryover horizon. Both are expression trees — see `marginal_expr`.

In [ ]:
surface = Surface(world.spec)
if not is_failure(post):
    pred = predict(surface, post, world.data, seed=0)
    print(pred.values.shape, pred.intervention.version)
    print("coverage of the noise-free mean by the 90% HDI:",
          np.mean([(np.quantile(pred.values[..., u, t], 0.05) <= world.mean[u, t] <= np.quantile(pred.values[..., u, t], 0.95)) for u in range(4) for t in range(30)]).round(2))

In [ ]:
if not is_failure(post):
    draws = pred.values.reshape(-1, *world.mean.shape)[:, 0, :]
    lo, hi = np.percentile(draws, [5, 95], axis=0)
    fig = curve_band(
        np.arange(world.mean.shape[1]), draws.mean(axis=0), lo, hi,
        label="fitted mean",
        title="The fit against the unit it was fitted on",
        subtitle="unit 0 — posterior mean with its 90% band, the noise-free truth, and the noisy data it saw",
        x_title="period", y_title="outcome",
    )
    curve_band(np.arange(world.mean.shape[1]), world.mean[0], label="truth", color=ORANGE, dash="dot", fig=fig)
    fig.add_scatter(x=np.arange(world.mean.shape[1]), y=world.panel.array("y")[0], mode="markers",
                    marker={"size": 7, "color": "#898781"}, name="observed", showlegend=True)
    caption(fig, "The band tracks the truth and not the noise, which is the whole difference "
                 "between a fit and an interpolation. The points it is missing are the "
                 "measurement error it was told to expect.")
    fig
from axiom.core import dimension, latex
print(dimension(marginal_expr(surface, "a")))
print(marginal(surface, world.theta, world.data, "a")[0, :4].round(4), marginal_total(surface, world.theta, world.data, "a")[0, :4].round(4))

`counterfactual_doses` applies an `Intervention` (set / scale / shift, optionally on a support
window) to the fitted dose arrays; `predict_marginal` gives per-draw marginal effects for a
chosen `MarginalHorizon` — `period` (same-period), `total` (over the carryover horizon), or
`shift` (the derivative of the windowed outcome w.r.t. a common shift of the support's doses).

In [ ]:
from axiom.core import Intervention, TimeWindow

cf = counterfactual_doses(surface, world.data, Intervention(doses={"a": 2.0}, mode="scale", window=TimeWindow(start=5, stop=10)))
print(cf["a"][0, 3:12].round(1), "<- scaled on [5,10) only")
if not is_failure(post):
    horizon: MarginalHorizon = "shift"
    pm = predict_marginal(surface, post, world.data, "a", horizon=horizon, support=TimeWindow(start=5, stop=10))
    print(pm.values.shape, pm.values.mean(axis=(0, 1))[0, 3:12].round(4))

## Unit invariance of a fit

Fit the same world expressed in cents (doses ×100): the shape `s` and carryover `lam` agree
within Monte-Carlo error, and the scale `k` differs by exactly the conversion factor.

In [ ]:
import pandas as pd

from axiom.core import D, Treatment
from axiom.data import Panel, RoleMap

frame = world.panel.frame
frame_cents = frame.assign(a=frame["a"] * 100)
roles = world.panel.roles
roles_cents = roles.model_copy(update={"treatments": {"a": Treatment(name="a", dimension=D.currency, unit="cents")}})
spec_cents = world.spec.model_copy(update={
    "treatments": (Treatment(name="a", dimension=D.currency, unit="cents"),),
    "kernels": {"a": HillKernel(reference_dose=5000.0)},
})
res_cents = fit(spec_cents, Panel(frame_cents, roles_cents), backend="laplace", draws=2000, seed=0)
pc = res_cents.posterior
if not (is_failure(post) or is_failure(pc)):
    table(
        [
            [name, round(post.summary(name).mean, 3), round(pc.summary(name).mean, 3)]
            for name in ("s_a", "lam_a", "k_a")
        ],
        headers=("parameter", "fitted in currency", "fitted in cents"),
    )

## What this bought you

A fit whose failure mode is a typed `Unverified` rather than a plausible posterior, a
prediction that carries its band, marginal effects that are the kernel's own closed-form
derivative, and a unit invariance you can check rather than assume.

`06-bands.ipynb` is where that band becomes the object every surface figure in the package is
required to draw.